In [2]:
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError
%load_ext google.cloud.bigquery

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


# **Entity Join**

In this notebook, we connect Snack and Theater to create a Snack_Service entity. We then connect Movie and Theater to create a Movie_Service entity.

In [1]:
%%bigquery
select *
from movie_stg.Snack
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,calories,calories_from_fat,total_fat,saturated_fat,trans_fat,polyunsat_fat,monounsat_fat,cholesterol,sodium,potassium,carbohydrates,dietary_fiber,sugars,protein,data_source,load_time
0,81,Mango Smoothie 16 fl oz,270.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,NaN,72.0,NaN,68.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
1,59,Apple Sauce,40.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,75.0,10.0,NaN,8.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
2,79,Banana Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,105.0,NaN,99.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
3,89,Strawberry Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102.0,NaN,93.0,NaN,amc,2024-02-07 03:53:10.283111+00:00
4,84,Peach Smoothie 24 fl oz,390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,0.0,104.0,NaN,101.0,NaN,amc,2024-02-07 03:53:10.283111+00:00


In [2]:
%%bigquery
select *
from movie_stg.Theater
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,id,name,address,city,state,zip,country,theaterChainName,longtitude,latitude,data_source,load_time
0,2907,Simsbury,530 Bushy Hill Road,Simsbury,CT,6070,USA,Hoyts,-72.863399,41.819424,github,2024-01-31 00:28:40.136547+00:00
1,872,West Nursery,1591 W. Nursery Road,Linthicum,MD,21090,USA,Hoyts,-76.677665,39.205236,github,2024-01-31 00:28:40.136547+00:00
2,312,FunAsia Cinema,1210 E. Belt Line Rd.,Richardson,TX,75081,USA,FunAsiA,-96.7037,32.9452,github,2024-01-31 00:28:40.136547+00:00
3,4441,Monaco Pictures,"370 The Bridge Street, NW",Huntsville,AL,35806,USA,Cinemark,-86.67353,34.718965,github,2024-01-31 00:28:40.136547+00:00
4,4976,Cinemark Conway Towne Centre,201 Skyline Drive,Conway,AR,72032,USA,Cinemark,-92.430747,35.109712,github,2024-01-31 00:28:40.136547+00:00


In [3]:
%%bigquery
select *
from movie_stg.Movie
limit 5

Query is running:   0%|          |

Downloading:   0%|          |

,tmdb_id,imdb_id,title,original_language,overview,release_date,genre_ids,duration,directors,writers,data_source,load_time
0,618219,None,Mosul,ar,"When ISIS took their homes, families and city,...",2019-09-04,"[10752, 18, 28]",<NA>,None,None,tmdb,2024-02-08 23:34:20.679895+00:00
1,1221388,None,From the Ashes,ar,"At a strict girls school, a mysterious fire bl...",2024-01-18,"[53, 18]",<NA>,None,None,tmdb,2024-02-08 23:34:20.679895+00:00
2,320956,None,The Cuckold,ar,The Cuckold is a controversial dramatic featur...,2009-08-11,[18],<NA>,None,None,tmdb,2024-02-08 23:34:20.679895+00:00
3,517814,None,Capernaum,ar,"Zain, a 12-year-old boy scrambling to survive ...",2018-09-20,[18],<NA>,None,None,tmdb,2024-02-08 23:34:20.679895+00:00
4,193457,None,A Cultural Film,ar,"To watch a pornographic film in Egypt, you nee...",2000-10-04,[35],<NA>,None,None,tmdb,2024-02-08 23:34:20.679895+00:00


# Generic Functions

These functions will be used by both Snack_Service and Movie_Service:

In [4]:
import json, datetime
from google.cloud import bigquery
bq_client = bigquery.Client()

def serialize_datetime(obj):
    if isinstance(obj, datetime.datetime):
        return obj.isoformat()
    raise TypeError("Type not serializable")

# removes the entries from the dictionary whose values are None
# this filter is needed for loading JSON into BQ
def remove_none_values(record):
    filtered_record = {}
    for field in record.keys():
        if record[field] != None:
            filtered_record[field] = record[field]
    return filtered_record


def create_table(table_id, schema):

    table = bigquery.Table(table_id, schema=schema)
    table = bq_client.create_table(table, exists_ok=True)
    print("Created table {}".format(table.table_id))


def load_records(table_id, schema, records):

    bq_client = bigquery.Client()

    # load records into staging table
    job_config = bigquery.LoadJobConfig(schema=schema, source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON, write_disposition='WRITE_TRUNCATE')
    table_ref = bigquery.table.TableReference.from_string(target_table_id)

    try:
        load_job = bq_client.load_table_from_json(records, table_ref, job_config=job_config)
        load_job.result()

        destination_table = bq_client.get_table(target_table_id)
        print("Loaded {} rows.".format(destination_table.num_rows))

        if load_job.errors:
            print('job errors:', load_job.errors)


    except Exception as e:
        print("Error inserting into BQ: {}".format(e))

# Movie_Service

Movies will have service_type as in-theaters if, when sorted by popularity (descending order), the movie has a release date within the last 6 weeks including this week. For each in-theater movie, assign a random theater.

In [12]:
%%bigquery
select count(*)
from
    (select m.tmdb_id, mh.popularity
    from movie_stg.Movie m join movie_stg.Movie_History mh
    on m.tmdb_id = mh.movie_id
    where m.release_date BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 6 WEEK) AND CURRENT_DATE()
    order by mh.popularity DESC)

Query is running:   0%|          |

Downloading:   0%|          |

,f0_
0,161


In [ ]:
from google.cloud import bigquery
bq_client = bigquery.Client()

ms_records = []
theater_ids = []

project_id = "auspicious-lead-382321"
stg_dataset_name = "movie_stg"
stg_table_name = "Movie_Service"
target_table_id = "{}.{}.{}".format(project_id, stg_dataset_name, stg_table_name)

schema = [bigquery.SchemaField("service_id", "STRING", mode="REQUIRED"),
          bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
          bigquery.SchemaField("service_type", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("theater_id", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
          bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
          ]

create_table(target_table_id, schema)

del schema[-1]

theater_sql = 'select id from {}.Theater'.format(stg_dataset_name)
query_job = bq_client.query(theater_sql)
# add the theater ids to a list
for row in query_job:
    theater_ids.append(row["id"])

# get all popular movies that have a release date within the last 3 weeks
intheaters_sql = ''' 
select tmdb_id as movie_id
from
    (select m.tmdb_id, mh.popularity
    from movie_stg.Movie m join movie_stg.Movie_History mh
    on m.tmdb_id = mh.movie_id
    where m.release_date BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 6 WEEK) AND CURRENT_DATE()
    order by mh.popularity DESC)
'''
query_job = bq_client.query(intheaters_sql)

index = 0

# for each in-theater movie, assign it a theater (aka theater_id)
for row in query_job:
    movie_id = row["movie_id"]
    
    # there are no more theaers to allocate
    if index >= len(theater_ids):
        break
    theater_id = theater_ids[index]
    service_id = str(uuid.uuid4())[:13]
    # assign a theater to the in-theater movie
    record = {"service_id":service_id, "movie_id": movie_id, "service_type": "in-theaters", "theater_id": theater_id, "data_source": "tmdb_github"}

    filtered_record = remove_none_values(record)

    ms_records.append(filtered_record)

    index += 1

load_records(target_table_id, schema, ms_records)

In [23]:
from google.cloud import bigquery
import random
import uuid
bq_client = bigquery.Client()

ms_records = []
theater_ids = []

project_id = "auspicious-lead-382321"
stg_dataset_name = "movie_stg"
stg_table_name = "Movie_Service"
target_table_id = "{}.{}.{}".format(project_id, stg_dataset_name, stg_table_name)

schema = [bigquery.SchemaField("service_id", "STRING", mode="REQUIRED"),
          bigquery.SchemaField("movie_id", "INTEGER", mode="REQUIRED"),
          bigquery.SchemaField("service_type", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("theater_id", "INTEGER", mode="NULLABLE"),
          bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
          bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
          ]

create_table(target_table_id, schema)

del schema[-1]

theater_sql = 'select id from {}.Theater'.format(stg_dataset_name)
query_job = bq_client.query(theater_sql)
# add the theater ids to a list
for row in query_job:
    theater_ids.append(row["id"])

# get all popular movies that have a release date within the last 3 weeks
intheaters_sql = ''' 
select tmdb_id as movie_id
from
    (select m.tmdb_id, mh.popularity
    from movie_stg.Movie m join movie_stg.Movie_History mh
    on m.tmdb_id = mh.movie_id
    where m.release_date BETWEEN DATE_SUB(CURRENT_DATE(), INTERVAL 6 WEEK) AND CURRENT_DATE()
    order by mh.popularity DESC)
'''
query_job = bq_client.query(intheaters_sql)

index = 0

# for each in-theater movie, assign it a 1-5 theaters (aka theater_id)
for row in query_job:
    movie_id = row["movie_id"]
    
    # there are no more theaers to allocate
    if index >= len(theater_ids):
        break
        
    # Determine a random number of theaters for this movie
    num_theaters = random.randint(1, 5)
    # Randomly select theater_ids without replacement to avoid duplicates
    selected_theater_ids = random.sample(theater_ids, num_theaters)    
    # assign number (1-5) of random theaters to the in-theater movie
    for theater_id in selected_theater_ids:
        service_id = str(uuid.uuid4())[:13]
        record = {"service_id":service_id, "movie_id": movie_id, "service_type": "in-theaters", "theater_id": theater_id, "data_source": "tmdb_github"}

        filtered_record = remove_none_values(record)

        ms_records.append(filtered_record)

    index += 1

load_records(target_table_id, schema, ms_records)

Created table Movie_Service
Loaded 479 rows.


## Primary Key

In [24]:
%%bigquery
alter table movie_stg.Movie_Service
  add primary key (service_id) not enforced

Query is running:   0%|          |

""


In [25]:
%%bigquery
select service_id, count(*) as duplicate_records
from movie_stg.Movie_Service
group by service_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,service_id,duplicate_records


## Foreign Keys

In [26]:
%%bigquery
alter table movie_stg.Movie_Service add foreign key (theater_id)
  references movie_stg.Theater (id) not enforced

Query is running:   0%|          |

""


In [27]:
%%bigquery
alter table movie_stg.Movie_Service add foreign key (movie_id)
  references movie_stg.Movie (tmdb_id) not enforced

Query is running:   0%|          |

""


In [29]:
%%bigquery
select count(*) as orphan_records
from movie_stg.Movie_Service
where movie_id not in (select tmdb_id from movie_stg.Movie)

Query is running:   0%|          |

Downloading:   0%|          |

,orphan_records
0,0


In [30]:
%%bigquery
select count(*) as orphan_records
from movie_stg.Movie_Service
where theater_id not in (select theater_id from movie_stg.Theater)

Query is running:   0%|          |

Downloading:   0%|          |

,orphan_records
0,0


# Snack_Service

Create the **Snack_Service** table and populate it with some snack records

In [ ]:
from google.cloud import bigquery
bq_client = bigquery.Client()

ss_records = []
snack_ids = []

project_id = "auspicious-lead-382321"
stg_dataset_name = "movie_stg"
stg_table_name = "Snack_Service"
target_table_id = "{}.{}.{}".format(project_id, stg_dataset_name, stg_table_name)

schema = [bigquery.SchemaField("theater_name", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("snack_id", "INTEGER", mode="NULLABLE"),
          bigquery.SchemaField("theater_id", "INTEGER", mode="NULLABLE"),
          bigquery.SchemaField("theater_chain_name", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("service_category", "STRING", mode="NULLABLE"),
          bigquery.SchemaField("data_source", "STRING", mode="REQUIRED"),
          bigquery.SchemaField("load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
          ]

create_table(target_table_id, schema)

del schema[-1]

snack_sql = 'select id from {}.Snack'.format(stg_dataset_name)
query_job = bq_client.query(snack_sql)

for row in query_job:
  snack_ids.append(row["id"])

theater_sql = '''
select theaterChainName, ARRAY_AGG(id) as theater_ids, ARRAY_AGG(name) as theater_names
from movie_stg.Theater
group by theaterChainName
'''

query_job = bq_client.query(theater_sql)

index = 0

for row in query_job:
    theater_chain = row["theaterChainName"]
    theater_ids= row["theater_ids"]
    theater_names = row["theater_names"]

    if index >= len(snack_ids):
        break
    snack_id = snack_ids[index]
    for i in range(len(theater_names)):
        record = {"theater_name": theater_names[i], "snack_id": snack_id, "theater_id": theater_ids[i], "theater_chain_name": theater_chain, "data_source": "amc_github"}
        filtered_record = remove_none_values(record)
        ss_records.append(filtered_record)
    index += 1

load_records(target_table_id, schema, ss_records)

## Primary Key

In [6]:
%%bigquery
alter table movie_stg.Snack_Service
  add primary key (snack_id, theater_id) not enforced

Query is running:   0%|          |

""


In [7]:
%%bigquery
select snack_id, theater_id, count(*) as duplicate_pk
from movie_stg.Snack_Service
group by snack_id, theater_id
having count(*) > 1

Query is running:   0%|          |

Downloading: |          |

,snack_id,theater_id,duplicate_pk


## Foreign Key

In [9]:
%%bigquery
alter table movie_stg.Snack_Service add foreign key (snack_id)
  references movie_stg.Snack (id) not enforced

Query is running:   0%|          |

""


In [10]:
%%bigquery
alter table movie_stg.Snack_Service add foreign key (theater_id)
  references movie_stg.Theater (id) not enforced

Query is running:   0%|          |

""


Check for foreign key violations

In [12]:
%%bigquery
select count(*) as orphan_records
from movie_stg.Snack_Service
where snack_id not in (select id from movie_stg.Snack)

Query is running:   0%|          |

Downloading:   0%|          |

,orphan_records
0,0


In [13]:
%%bigquery
select count(*) as orphan_records
from movie_stg.Snack_Service
where theater_id not in (select id from movie_stg.Theater)

Query is running:   0%|          |

Downloading:   0%|          |

,orphan_records
0,0
